In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import time

# -----------------------------
# Load Dataset
# -----------------------------
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

In [5]:
# small subset for speed
subset = torch.utils.data.Subset(train_dataset, range(5000))

train_loader = torch.utils.data.DataLoader(subset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)

In [6]:
# -----------------------------
# Model (same as Part 4)
# -----------------------------
input_size = 784
h1_size = 256
h2_size = 128
output_size = 10

W1 = torch.randn(input_size, h1_size, requires_grad=True)
b1 = torch.zeros(h1_size, requires_grad=True)

W2 = torch.randn(h1_size, h2_size, requires_grad=True)
b2 = torch.zeros(h2_size, requires_grad=True)

W3 = torch.randn(h2_size, output_size, requires_grad=True)
b3 = torch.zeros(output_size, requires_grad=True)

lr = 0.01
epochs = 2

In [8]:
# -----------------------------
# Step 1: Freeze earlier layers
# -----------------------------
W1.requires_grad = False
b1.requires_grad = False

W2.requires_grad = False
b2.requires_grad = False


# -----------------------------
# Step 2: Train only final layer
# -----------------------------
print("Training ONLY final layer...")

start_time = time.time()

for epoch in range(epochs):
    total_loss = 0

    for images, labels in train_loader:

        x = images.reshape(-1, 784)

        # forward
        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        loss = torch.nn.functional.cross_entropy(logits, labels)

        loss.backward()

        # update only last layer
        with torch.no_grad():
            W3 -= lr * W3.grad
            b3 -= lr * b3.grad

        W3.grad.zero_()
        b3.grad.zero_()

        total_loss += loss.item()

    print(f"[Frozen] Epoch {epoch+1}, Loss: {total_loss:.4f}")

frozen_time = time.time() - start_time

Training ONLY final layer...
[Frozen] Epoch 1, Loss: 13981.4157
[Frozen] Epoch 2, Loss: 12789.5239


In [9]:
# -----------------------------
# Step 3: Unfreeze all layers
# -----------------------------
W1.requires_grad = True
b1.requires_grad = True

W2.requires_grad = True
b2.requires_grad = True

In [11]:
# -----------------------------
# Step 4: Train full network
# -----------------------------
print("\nTraining FULL network...")

start_time = time.time()

for epoch in range(epochs):
    total_loss = 0

    for images, labels in train_loader:

        x = images.reshape(-1, 784)

        # forward
        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        loss = torch.nn.functional.cross_entropy(logits, labels)

        loss.backward()

        # update all
        with torch.no_grad():
            W1 -= lr * W1.grad
            b1 -= lr * b1.grad

            W2 -= lr * W2.grad
            b2 -= lr * b2.grad

            W3 -= lr * W3.grad
            b3 -= lr * b3.grad

        # zero grads
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()
        W3.grad.zero_()
        b3.grad.zero_()

        total_loss += loss.item()

    print(f"[Unfrozen] Epoch {epoch+1}, Loss: {total_loss:.4f}")

full_time = time.time() - start_time


Training FULL network...
[Unfrozen] Epoch 1, Loss: 754.8504
[Unfrozen] Epoch 2, Loss: 549.3447


In [12]:
# -----------------------------
# Step 5: Evaluation
# -----------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        x = images.reshape(-1, 784)

        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total


# -----------------------------
# Final Comparison
# -----------------------------
print("\n--- Results ---")
print(f"Final Accuracy: {accuracy:.2f}%")
print(f"Frozen Training Time: {frozen_time:.2f} sec")
print(f"Full Training Time: {full_time:.2f} sec")


--- Results ---
Final Accuracy: 64.74%
Frozen Training Time: 2.56 sec
Full Training Time: 2.67 sec
